# API Example

This notebook demonstrates end-to-end API calls. It includes use of various tools including connecting Longhorn, NEO4j, and custom services using Kubernetes API for distributed computing.

In [ ]:
import requests, json
import pandas as pd
from IPython.core.display import HTML

# Adjust display settings
pd.set_option('display.max_colwidth', None)  # No truncation of column content
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)        # Adjust width automatically to fit content

analytics_url = "http://analytics.prod.dart.prv.vai-compute.com/analytics/v1"
scheduler_url = "http://scheduler.prod.dart.prv.vai-compute.com/scheduler/v1"
crawler_url = "http://crawler.prod.dart.prv.vai-compute.com/crawler/v1"
transformer_url = "http://transformer.prod.dart.prv.vai-compute.com/transformer/v1"
gat_url = "http://gat.prod.dart.prv.vai-compute.com/gat/v1"
nets_url = "http://nets.prod.dart.prv.vai-compute.com/nets/v1"
routes_url = "http://routes.prod.dart.prv.vai-compute.com/routes/v1"

## 1. Storage

### Persistent Volume Claims (PVCs)

First, you need storage mounted to your services. We will do this ahead in anticipation that we will need this. In this step, we'll create a PVC, list all PVCs.

In [ ]:
# Create PVC for logs
response = requests.post(f"{scheduler_url}/longhorn/create_pvc", json={
    "name": "auto-test-0",
    "type": "sata",
    "size": "1Mi",
    "namespace": "vai-rd",
})

# Print the response
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "message": "PVC created successfully"
}


You can see all of the PV's that have been created by doing the following:

In [28]:
# List PVs
response = requests.post(f"{scheduler_url}/longhorn/pvs", json={"namespace": "lonhorn-system"})
pd.DataFrame(response.json())

,accessModes,created,name,reclaimPolicy,status,storage,storageClass
0,[ReadWriteOnce],"Fri, 22 Nov 2024 08:14:56 GMT",alex-panopticon-models-recovered,Retain,Bound,80Gi,longhorn-nvme
1,[ReadWriteMany],"Tue, 25 Mar 2025 06:40:43 GMT",analytics,Delete,Available,1Gi,standard
2,[ReadWriteMany],"Tue, 17 Dec 2024 19:54:32 GMT",asset-ingest-cache,Retain,Bound,100Gi,longhorn-nvme
3,[ReadWriteMany],"Mon, 07 Apr 2025 20:03:29 GMT",auto-logs-0,Delete,Available,20Gi,standard
4,[ReadWriteMany],"Tue, 29 Apr 2025 21:05:39 GMT",auto-models-0,Delete,Available,100Gi,standard
...,...,...,...,...,...,...,...
303,[ReadOnlyMany],"Mon, 25 Nov 2024 21:43:51 GMT",warc-processor-data-backup,Retain,Bound,120Ti,None
304,[ReadWriteMany],"Mon, 25 Nov 2024 21:44:31 GMT",warc-processor-data-backup-rwx,Retain,Bound,120Ti,None
305,[ReadOnlyMany],"Fri, 24 May 2024 17:42:18 GMT",warc-processor-data-vaicore,Retain,Bound,120Ti,None
306,[ReadOnlyMany],"Thu, 30 May 2024 18:16:11 GMT",warc-processor-data-vaicore-staging,Retain,Bound,120Ti,None


In addition, you can see all of the PVC's associated with each pods by doing the following:

In [29]:
# List PVCs
response = requests.post(f"{scheduler_url}/longhorn/pvcs", json={'namespace': 'vai-rd'})
pd.DataFrame(response.json())

,name,pods,status,storage
0,analytics,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi
1,auto-data-0,"[{'node_name': 'janus', 'pod_name': 'auto-anal...",Bound,20Gi
2,auto-dist-0,"[{'node_name': 'janus', 'pod_name': 'auto-anal...",Bound,1Gi
3,auto-logs-0,"[{'node_name': 'janus', 'pod_name': 'auto-anal...",Bound,20Gi
4,auto-models-0,"[{'node_name': 'janus', 'pod_name': 'auto-anal...",Bound,100Gi
5,crawler,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi
6,gat,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi
7,jupyter,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi
8,nets,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi
9,routes,"[{'node_name': 'janus', 'pod_name': 'auto-jupy...",Bound,1Gi


If you need to delete the PV's and PVC's, you can use the following

In [21]:
# Delete PV and PVC
response = requests.delete(f"{scheduler_url}/longhorn/delete_pv_and_pvc", json={
    "name": "auto-test-0",
    "namespace": "vai-rd"
})
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "message": "PV and PVC deleted successfully"
}


# 2. Data (Webpage Crawl)

Now lets go into an example. First, we data. Let's create a graph to store it. We will need nodes and edges.

## Parse Domain

We will employ DOM tree parsing techniques to extract textual content.

** Importantly, this extraction is carried out alongside the scraping process, allowing for the immediate and orderly storage of data in our database.

In [19]:
parsed_data = requests.post(f"{crawler_url}/parse", json={
    "urls": [
        "virtuousai.com",
        # "sarus.tech",
        # "modulos.ai",
        # "sherpa.ai",
        # "theodi.org/organisations/profile/trilateral-research/",
        # "clearbox.ai",
        # "chatterbox.co",
        # "learn.microsoft.com/en-us/azure/cloud-adoption-framework/ready/landing-zone/",
        # "ibm.com/products/watson-studio",
        # "infosys.com/industries/insurance/insights/automation-improve-insurer-client-engagement.html"
    ]
}).json()

nodes, edges = parsed_data['nodes'], parsed_data['edges']

In [ ]:
# Send the POST request
response = requests.post(f"{analytics_url}/vertical_tree", json={
    "nodes": nodes,
    "edges": edges,
    "title": "Tree Plot Example",
    "node_size": 20,
    "node_color": "lightblue",
    "edge_color":  "gray"
})
response.raise_for_status()  # Raise an exception for HTTP errors
print(json.dumps(response.json(), indent=4))

HTML(f'<iframe src="{analytics_url}/vertical_tree" width="800" height="600"></iframe>')

{
    "figure": "{\"data\":[{\"hoverinfo\":\"none\",\"line\":{\"color\":\"gray\",\"width\":1},\"mode\":\"lines\",\"x\":[-9.754999999999999,-7.754999999999999,null,0.0,-4.504999999999999,null,-3.754999999999999,-1.754999999999999,null,0.0,5.145000000000001,null,20.745,17.745,null,17.745,14.245000000000001,null,17.745,15.245000000000001,null,17.745,16.245,null,17.745,17.245,null,17.745,18.245,null,17.745,19.245,null,17.745,20.245,null,17.745,21.245,null,23.745,22.245,null,23.745,23.245,null,23.745,24.245,null,23.745,25.245,null,-10.254999999999999,0.0,null,0.0,-9.254999999999999,null,-9.254999999999999,-9.754999999999999,null,-9.754999999999999,-11.754999999999999,null,-9.754999999999999,-10.754999999999999,null,-9.754999999999999,-9.754999999999999,null,-9.754999999999999,-8.754999999999999,null,-9.254999999999999,-8.754999999999999,null,-4.504999999999999,-5.754999999999999,null,-5.754999999999999,-6.754999999999999,null,-5.754999999999999,-5.754999999999999,null,-5.754999999999999,-4.

Then we add the nodes to neo4j.

In [25]:
updated_nodes = requests.post(f"{scheduler_url}/neo4j/create_nodes", json={"nodes": nodes})

In [ ]:
nodes_list = requests.post(f"{scheduler_url}/neo4j/list_nodes", json={"node_type": "Elements"})
pd.DataFrame(nodes_list.json())

,backlink,breadth,created_at,depth,embedding,leaf,link,local_id,name,node_id,tag,uuid
0,https://content.virtuousai.com/from-raw-to-relevant,144,2025-05-02T17:58:00.392171,8,[],True,virtuousai.com,779,Raw to Relevant Data Roadmap,51,a,ed3a73a8-851b-4446-ae16-462ff331d089
1,https://20495786.fs1.hubspotusercontent-na1.net/hubfs/20495786/1%20pagers/VirtueStack%20by%20VirtousAI.pdf,146,2025-05-02T17:58:00.430676,8,[],True,virtuousai.com,781,VirtueStackâ¢ Overview,52,a,45ce46c3-c6a5-411e-aa8d-a1b5a556322e
2,https://content.virtuousai.com/ai-unboxed-leaders-guide-to-ai-infrastructure,147,2025-05-02T17:58:00.473209,8,[],True,virtuousai.com,782,AI Adoption Plan,53,a,200a139e-4074-4ae2-9330-d17e8f0e86f2
3,https://virtuousai.com/#pricing,148,2025-05-02T17:58:00.514085,8,[],True,virtuousai.com,783,Pricing,54,a,f5dc347e-0d72-4c9a-a13e-4c6c113a64d8
4,https://virtuousai.com/#about,149,2025-05-02T17:58:00.548316,8,[],True,virtuousai.com,784,About Us,55,a,772ef60d-729b-4a05-9ff1-657bf7e8ab1e
...,...,...,...,...,...,...,...,...,...,...,...,...
90,https://content.virtuousai.com/book-a-demo,52,2025-05-02T17:58:00.239137,6,[],True,virtuousai.com,771,request a demo,153,a,e0a8b715-14c6-4212-8f48-33243d0e3820
91,,53,2025-05-02T17:58:00.278475,6,[],False,virtuousai.com,774,,154,div,08074ea3-0f2e-4305-b149-0b5314cd5e7a
92,,85,2025-05-02T17:58:00.316204,7,[],False,virtuousai.com,777,,155,div,e16797f2-c748-4bfd-8516-aac6a00cfcf0
93,/blog/,143,2025-05-02T17:58:00.354810,8,[],True,virtuousai.com,778,Blog,156,a,740c17b4-866b-41b0-a30c-90f3a25fe380


Then the edges are added.

In [ ]:
updated_edges = requests.post(f"{scheduler_url}/neo4j/create_edges", json={"edges": edges})

In [ ]:
response = requests.post(f"{scheduler_url}/neo4j/list_edges", json={"edge_type": "CONNECTED_TO"})
pd.DataFrame(response.json())

,created_at,link,source_id,source_uuid,target_id,target_uuid
0,2025-05-02T19:56:12.796446,virtuousai.com,70,ff17a325-9d3f-4ce4-b053-cc80c4baf30c,71,c63467e9-c12a-44e7-ad3e-bae17fbe79a3
1,2025-05-02T19:56:12.908282,virtuousai.com,70,ff17a325-9d3f-4ce4-b053-cc80c4baf30c,72,1b1b48ee-8f2b-457b-87aa-331b5a594935
2,2025-05-02T19:56:13.029780,virtuousai.com,70,ff17a325-9d3f-4ce4-b053-cc80c4baf30c,73,c0319d96-2d46-439a-896d-4083f598d119
3,2025-05-02T19:56:13.305791,virtuousai.com,70,ff17a325-9d3f-4ce4-b053-cc80c4baf30c,74,f701d91c-451c-4817-87b0-fcf5fc764abf
4,2025-05-02T19:56:20.644827,sarus.tech,75,371336b6-1aa5-4739-8666-39899bbc1305,76,b2842b0d-1b19-42c3-80ad-2137d4fe6a56
...,...,...,...,...,...,...
86,2025-05-02T19:56:12.311328,virtuousai.com,155,e16797f2-c748-4bfd-8516-aac6a00cfcf0,54,f5dc347e-0d72-4c9a-a13e-4c6c113a64d8
87,2025-05-02T19:56:12.446516,virtuousai.com,155,e16797f2-c748-4bfd-8516-aac6a00cfcf0,55,772ef60d-729b-4a05-9ff1-657bf7e8ab1e
88,2025-05-02T19:56:12.566332,virtuousai.com,155,e16797f2-c748-4bfd-8516-aac6a00cfcf0,68,1d166f09-8b98-4e5c-97d5-044928122df0
89,2025-05-02T19:56:12.683788,virtuousai.com,155,e16797f2-c748-4bfd-8516-aac6a00cfcf0,69,6722ec32-eedc-45e0-b2c3-0bf2074ca3a2


# 3.1. Machine Learning

## 3.1. Text Data

Now that we have all of the nodes and edges, we need to embed. Lets embed all of the chunked graph.


In [ ]:
response = requests.post(
    f"{nets_url}/train/textual",
    json={
        "org": "rory",
        "project": "example_text",
        "file_paths": ["/mnt/data/agtech/test_project/labels.csv"],
        "params": {
            "learning_rate": 0.001,
            "epochs": 1,
            "batch_size": 16
        },
        "dtypes": {
            "x": {
                "location_1": "textual"
            },
            "y": {
                "crop_2": "categorical"
            }
        }
    }
)
print("Train Response:", response.status_code, response.json())
filename = response.json()

Train Response: 200 {'filename': 'TEXT_1746687541_031d7d6d2f9447baa419a57f5e351ade.pth'}


In [54]:
response = requests.post(
    "http://nets.prod.dart.prv.vai-compute.com/nets/v1/extract_model_meta",
    json={
        "org": "rory",
        "project": "example_text",
        "unique_filename": "TEXT_1746687541_031d7d6d2f9447baa419a57f5e351ade.pth"
    }
)
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "dtypes": {
        "x": {
            "location_1": {
                "type": "TEXT"
            }
        },
        "y": {
            "crop_2": {
                "type": "CATEGORY"
            }
        }
    },
    "file_paths": [
        "/mnt/data/agtech/test_project/labels.csv"
    ],
    "optimizer_state_present": true,
    "params": {
        "batch_size": 16,
        "epochs": 1,
        "learning_rate": 0.001,
        "text_model_index": 0
    },
    "process_metadata": {
        "elapsed_time": 6.435892820358276,
        "epoch": 1,
        "process_id": 576,
        "start_time": "2025-05-07T23:59:00",
        "status": "complete"
    },
    "status": "complete"
}


Now lets try some inferences.

In [217]:
response = requests.post(f"{nets_url}/predict/textual", json={
    "org": "rory",
    "project": "example_text",
    "filename": "TEXT_1746687541_031d7d6d2f9447baa419a57f5e351ade.pth",
    "inputs": {
        "location_1": [
            "This is the first example sentence.",
            "Here is another example of text input."
        ]
    }
})
pd.DataFrame(response.json())

,crop_2_harvested-crop,crop_2_healthy-crop,crop_2_no-crop,crop_2_stressed-crop
0,0.377596,-0.057859,0.166713,0.652911
1,0.382712,-0.100981,0.105748,0.584032


Now lets explain these predictions.

In [ ]:
response = requests.post(f"{nets_url}/explain/textual", json={
    "org": "rory",
    "project": "example_text",
    "filename": "TEXT_1746687541_031d7d6d2f9447baa419a57f5e351ade.pth",
    "inputs": {
        "location_1": [
            "This is the first example sentence.",
            "Here is another example of text input."
        ]
    }
})
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "expected_value": {
        "crop_2_harvested-crop": 0.3619523346424103,
        "crop_2_healthy-crop": 0.5240757465362549,
        "crop_2_no-crop": 0.4481528401374817,
        "crop_2_stressed-crop": -0.12813158333301544
    },
    "shap_values": {
        "0": {
            "crop_2_harvested-crop": {
                ".": 0.0,
                "[CLS]": 0.0,
                "[PAD]": 0.0,
                "[SEP]": 0.0,
                "example": 0.0,
                "first": 0.0,
                "is": 0.0,
                "sentence": 0.0,
                "the": 0.0,
                "this": 0.0
            },
            "crop_2_healthy-crop": {
                ".": 0.0,
                "[CLS]": 0.0,
                "[PAD]": 0.0,
                "[SEP]": 0.0,
                "example": 0.0,
                "first": 0.0,
                "is": 0.0,
                "sentence": 0.0,
                "the": 0.0,
                "this": 0.0
            },
            "crop_2_no-crop": {
  

We also have chatbot methods.

In [ ]:
import requests
response = requests.post(f"{routes_url}/gpt-4/prompt", json={
    "request_type": "step_by_step",
    "prompt": "how to bake a cake?"
})
data = response.json()
print(data["steps"])  # Output: <class 'list'>


## 3.2. Categorical Data

To test categorical data, we can just cluster the data. E.g. take the embeddings and create 10 groups that represent categories.

** note: this is placeholder for more practical example.

First we need to train a model

In [48]:
response = requests.post(f"{nets_url}/train/categorical", json={
    "org": "rory",
    "project": "example_categorical",
    "file_paths": ["/mnt/data/agtech/test_project/labels.csv"],
    "params": {
        "learning_rate": 0.01,
        "epochs": 20,
        "batch_size": 32
    },
    "dtypes": {
        "x": {
            "location_1": "categorical",
            "location_2": "categorical"
        },
        "y": {
            "crop_2": "categorical"
        }
    }
})
print("Train Response:", response.status_code, response.json())
filename = response.json().get("filename")

Train Response: 200 {'filename': 'CATEGORY_1750189201_41a05936935f468b87bc9a2120cbeb3b.pth', 'message': 'Check logs at https://nets.prod.dart.prv.vai-compute.com/nets/v1/'}


In [53]:
response = requests.post(
    "http://nets.prod.dart.prv.vai-compute.com/nets/v1/extract_model_meta",
    json={
        "org": "rory",
        "project": "example_categorical",
        "unique_filename": "CATEGORY_1750189201_41a05936935f468b87bc9a2120cbeb3b.pth"
    }
)
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "dtypes": {
        "x": {
            "location_1": {
                "type": "CATEGORY"
            },
            "location_2": {
                "type": "CATEGORY"
            }
        },
        "y": {
            "crop_2": {
                "type": "CATEGORY"
            }
        }
    },
    "file_paths": [
        "/mnt/data/agtech/test_project/labels.csv"
    ],
    "optimizer_state_present": true,
    "params": {
        "batch_size": 32,
        "epochs": 20,
        "learning_rate": 0.01
    },
    "process_metadata": {
        "elapsed_time": 1.2304127216339111,
        "epoch": 20,
        "process_id": 24,
        "start_time": "2025-06-17T12:40:01",
        "status": "initialized"
    },
    "status": "complete"
}


Then we can predict.

In [ ]:
response = requests.post(f"{nets_url}/predict/categorical", json={
    "org": "rory",
    "project": "example_categorical",
    "filename": "CATEGORY_1750109222_657a2459c4244bcab370744fa13f5a13.pth",
    "inputs": {
        "location_1": ["California", "Iowa"],
        "location_2": ["Monterey County", "Osceola County"]
    }
})
pd.DataFrame(response.json())

,crop_2_harvested-crop,crop_2_healthy-crop,crop_2_no-crop,crop_2_stressed-crop
0,0.188539,0.027204,-0.100906,0.345189
1,0.226197,0.457657,-0.441006,0.421969


Lastly, we can explain.

In [ ]:
response = requests.post(f"{nets_url}/explain/categorical", json={
    "org": "rory",
    "project": "example_categorical",
    "filename": "CATEGORY_1750109222_657a2459c4244bcab370744fa13f5a13.pth",
    "inputs": {
        "location_1": ["California", "Iowa"],
        "location_2": ["Monterey County", "Osceola County"]
    }
})
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "expected_value": {
        "crop_2_harvested-crop": 0.20736780762672424,
        "crop_2_healthy-crop": 0.2424306720495224,
        "crop_2_no-crop": -0.2709563076496124,
        "crop_2_stressed-crop": 0.38357919454574585
    },
    "shap_values": {
        "0": {
            "crop_2_harvested-crop": {
                "location_1_California": 0.1484975665807724,
                "location_1_Iowa": -0.03022732026875019,
                "location_2_Madera County": 0.0,
                "location_2_Monterey County": 0.017358331009745598,
                "location_2_Osceola County": -0.15445779263973236
            },
            "crop_2_healthy-crop": {
                "location_1_California": -0.11647865921258926,
                "location_1_Iowa": -0.06690242141485214,
                "location_2_Madera County": 0.0,
                "location_2_Monterey County": -0.05150636285543442,
                "location_2_Osceola County": 0.01966072805225849
            },
            "crop_

## 3.3. Image Data

Now we will try for images. First we need an image model.

In [ ]:
response = requests.post(
    f"{nets_url}/train/image",
    json={
        "org": "rory",
        "project": "example_image",
        "file_paths": ["/mnt/data/agtech/test_project/labels.csv"],
        "params": {
            "learning_rate": 0.1,
            "epochs": 100,
            "batch_size": 16
        },
        "dtypes": {
            "x": {
                "image_link": "image"
            },
            "y": {
                "crop_2": "categorical"
            }
        }
    }
)
print("Train Response:", response.status_code, response.json())
filename = response.json().get("filename")


Train Response: 200 {'filename': 'IMAGE_1748542583_eaae97d9533944388d3886afe0d231b2.pth', 'message': 'Check logs at https://nets.prod.dart.prv.vai-compute.com/nets/v1/'}


In [55]:
response = requests.post(
    "http://nets.prod.dart.prv.vai-compute.com/nets/v1/extract_model_meta",
    json={
        "org": "rory",
        "project": "example_image",
        "unique_filename": "IMAGE_1748542583_eaae97d9533944388d3886afe0d231b2.pth"
    }
)
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "dtypes": {
        "x": {
            "image_link": {
                "type": "IMAGE"
            }
        },
        "y": {
            "crop_2": {
                "type": "CATEGORY"
            }
        }
    },
    "file_paths": [
        "/mnt/data/agtech/test_project/labels.csv"
    ],
    "optimizer_state_present": true,
    "params": {
        "batch_size": 16,
        "epochs": 100,
        "learning_rate": 0.001
    },
    "process_metadata": {
        "elapsed_time": 10762.396482467651,
        "epoch": 100,
        "process_id": 683,
        "start_time": "2025-05-29T11:19:00",
        "status": "complete"
    },
    "status": "complete"
}


Next, we need to run infereneces.

In [12]:
response = requests.post(f"{nets_url}/predict/image", json={
    "org": "rory",
    "project": "example_image",
    "filename": "IMAGE_1748542583_eaae97d9533944388d3886afe0d231b2.pth",
    "inputs": {
        "image_link": [
            "/mnt/data/_file_d_1-fiJh-uMzGR9gF5DMKp9UblSV9DPDz0__view.jpeg",
            # "/mnt/data/_file_d_147IeyLrnJb-FkbOwUqFPki9QxEX0e06i_view.jpeg"
        ]
    }
})
if response.status_code == 200:
    predictions = response.json()
    pd.DataFrame(predictions)
else:
    print(f"Request failed: {response.status_code}\nContent: {response.content}")
    predictions = None

Request failed: 202
Content: b'{"message":"Prediction is running in the background. Check logs at https://nets.prod.dart.prv.vai-compute.com/nets/v1/"}\n'


Lastly, we need to explain the inferences.

In [ ]:
response = requests.post(f"{nets_url}/explain/image", json={
    "org": "rory",
    "project": "example_image",
    "filename": "IMAGE_1748542583_eaae97d9533944388d3886afe0d231b2.pth",
    "inputs": {
        "image_link": [
            "/mnt/data/_file_d_1-fiJh-uMzGR9gF5DMKp9UblSV9DPDz0__view.jpeg"
        ]
    }
})
print("Explain Response:", response.status_code, response.json())

Explain Response: 202 {'message': 'Explanation is running in the background. Check logs at https://nets.prod.dart.prv.vai-compute.com/nets/v1/'}


## 3.5. Graph Networks 

The formula is content to embedding. Embedding to embedding (as one unified list of mixed modal dat). We use Graph Attention Network (GAT).

First we need to train a model. We will use hyperopt because train has a lot of parameters.

In [3]:
response = requests.post(
    f"https://gat.prod.dart.prv.vai-compute.com/gat/v1/optuna_hyperopt",
    json={
        "org": "joor",
        "project": "brand_link_prediction",
        "epochs":3,
        "num_trials":3
    }
)
# response.raise_for_status()  # Raise an HTTPError for bad responses (4xx and 5xx)
# Print the response in a formatted JSON structure
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "storage_name": "sqlite:///joor_brand_link_prediction_study.db",
    "study_name": "joor_brand_link_prediction_study"
}


Now, check the status of the model to see if training has completed

In [70]:
# Send a GET request to the /check_study_state endpoint
response = requests.post(
    "https://gat.prod.dart.prv.vai-compute.com/gat/v1/check_study_state",
    json={
        "study_name": "joor_brand_link_prediction_study",
        "storage_name": "sqlite:///joor_brand_link_prediction_study.db"
    }
)

# Check if the request was successful
if response.status_code == 200:
    # Print the study state in a formatted JSON structure
    print(json.dumps(response.json(), indent=4, sort_keys=True))
else:
    # Print an error message if the request failed
    print(f"Failed to retrieve study state: {response.status_code}")
    print(response.text)

Failed to retrieve study state: 500
{"error":"Failed to execute check_study_state_endpoint: 'Record does not exist.'","traceback":"Traceback (most recent call last):\n  File \"/home/gat/src/scheduler/utils/api_helper.py\", line 48, in wrapper\n    return func(*args, **kwargs)\n  File \"/home/gat/src/gat/api.py\", line 635, in check_study_state_endpoint\n    study = optuna.load_study(study_name=study_name, storage=storage_name)\n  File \"/home/gat/.gat/gat-xDB_yiz8-3.10/lib/python3.10/site-packages/optuna/_convert_positional_args.py\", line 134, in converter_wrapper\n    return func(**kwargs)  # type: ignore[call-arg]\n  File \"/home/gat/.gat/gat-xDB_yiz8-3.10/lib/python3.10/site-packages/optuna/study/study.py\", line 1385, in load_study\n    study = Study(study_name=study_name, storage=storage, sampler=sampler, pruner=pruner)\n  File \"/home/gat/.gat/gat-xDB_yiz8-3.10/lib/python3.10/site-packages/optuna/study/study.py\", line 87, in __init__\n    study_id = storage.get_study_id_from_na

Next, we can reuse the model for any inference (prediction or classification).

In [5]:
response = requests.post(
    f"{gat_url}/predict",
    json={
        "org": "joor",
        "project": "brand_link_prediction",
        "filename": "GAT_1748036419_792139589ff5429bb7213378da8b8639_2.pth"
    }
)

# Print the response
print(json.dumps(response.json(), indent=4, sort_keys=True))

{
    "status": "check logs at https://gat.prod.dart.prv.vai-compute.com/gat/v1/"
}


Lastly we can use the attention weights to be able to trace the importance.

In [ ]:
# Example: Explain the predictions of a trained model using pairwise XAI
import requests

# Send the POST request
response = requests.post(
    f"{gat_url}/explain",
    json={
        "org": "joor",
        "project": "brand_link_prediction",
        "filename": "GAT_1746135838_fcd93a5a6dca443c9cb4c7ec3fbdeb78.pth"
    }
)

# Print the response
print(json.dumps(response.json(), indent=4, sort_keys=True))

# 5. Kubernetes Commands

1. pods: a single instance of a running process with a lifecycle, containing 1+ service
2. service: set of pods providing endpoints for load balancing and communication regardless of pod lifecycle
3. deployment: manages the deployment and scaling of a set of pods

#### Deploy/Serve
- **Deploy using Helm**: `source bin/_kube && redeploy_helm_chart ${REPO}`

#### Validation
1. **Check Deployments**: `kubectl get deployments --all-namespaces`
2. **Describe Deployment**: `kubectl describe deployment ${REPO} -n ${REPO}`
3. **Check Services**: `kubectl get services --all-namespaces`
4. **List Helm Charts**: `helm list --all-namespaces`

#### Additional Commands
1. **Get URL**:  
   `kubectl get configmap argocd-cm -n argocd -o jsonpath='{.data.url}'`
2. **Get Secret Key**:  
   `kubectl get secret argocd-secret -n argocd -o json | jq -r '.data["admin.password"]' | base64 --decode`
3. **Get Config Maps**:  
   `kubectl get configmap argocd-rbac-cm -n argocd -o yaml`
4. **List Container Images**:  
   `kubectl get pods -A -o jsonpath="{.items[*].spec.containers[*].image}" | tr -s '[[:space:]]' '\n' | sort | uniq`
5. **Port Forward**:  
   `kubectl port-forward svc/argocd-server -n argocd 8080:443`

#### Removal
1. **Delete Deployment**: `kubectl delete deployment ${REPO} -n <namespace>`
2. **Uninstall Helm Release**: `helm uninstall ${REPO} -n <namespace>`
3. **Delete Service**: `kubectl delete service ${REPO} -n <namespace>`
4. **Delete Pod**: `kubectl delete pod <pod-name> -n <namespace>`